In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
!pip install kaggle efficientnet_pytorch onnx onnxruntime

In [ ]:
# Setup Kaggle & download dataset
import os
from google.colab import files

uploaded = files.upload()  # upload kaggle.json
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
os.rename('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

DATASET_DIR = '/content/drive/MyDrive/realface/dataset'

if not os.path.exists(DATASET_DIR):
    os.makedirs(DATASET_DIR, exist_ok=True)
    !kaggle datasets download -d troykueh/real-vs-fake-faces-stylegan3 -p /content/kaggle_tmp
    !unzip -q /content/kaggle_tmp/*.zip -d {DATASET_DIR}
    print('Dataset downloaded and extracted to Drive')
else:
    print('Dataset already exists, skipping download')

In [ ]:
# Sampling 5000 real & 5000 fake
import random
import glob

REAL_DIR = os.path.join(DATASET_DIR, 'real')
FAKE_DIR = os.path.join(DATASET_DIR, 'fake')

EXTS = ('*.jpg', '*.jpeg', '*.png')

def gather_images(folder):
    paths = []
    for ext in EXTS:
        paths.extend(glob.glob(os.path.join(folder, '**', ext), recursive=True))
    return paths

random.seed(42)
real_paths = random.sample(gather_images(REAL_DIR), 5000)
fake_paths = random.sample(gather_images(FAKE_DIR), 5000)

print(f'real: {len(real_paths)} | fake: {len(fake_paths)}')

In [ ]:
# EDA
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# --- distribusi kelas ---
print(f'Class distribution  ->  real: {len(real_paths)}  |  fake: {len(fake_paths)}')

# --- ukuran gambar ---
def image_sizes(paths, n=500):
    sample = random.sample(paths, min(n, len(paths)))
    sizes = [Image.open(p).size for p in sample]  # (W, H)
    return np.array(sizes)

sizes = image_sizes(real_paths + fake_paths)
print(f'Width  -> min: {sizes[:,0].min()}  max: {sizes[:,0].max()}  mean: {sizes[:,0].mean():.1f}')
print(f'Height -> min: {sizes[:,1].min()}  max: {sizes[:,1].max()}  mean: {sizes[:,1].mean():.1f}')

# --- visualisasi 5 real + 5 fake ---
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Sample Images  (top: real | bottom: fake)', fontsize=13)

for i, ax in enumerate(axes[0]):
    ax.imshow(Image.open(real_paths[i]).convert('RGB'))
    ax.set_title('real'); ax.axis('off')

for i, ax in enumerate(axes[1]):
    ax.imshow(Image.open(fake_paths[i]).convert('RGB'))
    ax.set_title('fake'); ax.axis('off')

plt.tight_layout()
plt.show()

# --- mean & std pixel per channel RGB ---
def compute_mean_std(paths, n=1000, size=(128, 128)):
    sample = random.sample(paths, min(n, len(paths)))
    pixels = []
    for p in sample:
        img = np.array(Image.open(p).convert('RGB').resize(size), dtype=np.float32) / 255.0
        pixels.append(img.reshape(-1, 3))
    pixels = np.concatenate(pixels, axis=0)
    return pixels.mean(axis=0), pixels.std(axis=0)

mean, std = compute_mean_std(real_paths + fake_paths)
print(f'Mean RGB -> R: {mean[0]:.4f}  G: {mean[1]:.4f}  B: {mean[2]:.4f}')
print(f'Std  RGB -> R: {std[0]:.4f}  G: {std[1]:.4f}  B: {std[2]:.4f}')

In [ ]:
# Preprocessing & DataLoader
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split

all_paths  = real_paths + fake_paths
all_labels = [1] * len(real_paths) + [0] * len(fake_paths)

X_train, X_tmp, y_train, y_tmp = train_test_split(
    all_paths, all_labels, test_size=0.30, stratify=all_labels, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=42
)
print(f'train: {len(X_train)}  val: {len(X_val)}  test: {len(X_test)}')

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class RealFaceDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths     = paths
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img   = Image.open(self.paths[idx]).convert('RGB')
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return self.transform(img), label

train_ds = RealFaceDataset(X_train, y_train, train_tf)
val_ds   = RealFaceDataset(X_val,   y_val,   eval_tf)
test_ds  = RealFaceDataset(X_test,  y_test,  eval_tf)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f'Batches  ->  train: {len(train_loader)}  val: {len(val_loader)}  test: {len(test_loader)}')

In [ ]:
# Setup Model
import torch.nn as nn
from torchvision import models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(in_features, 1)
)

model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,}')

In [ ]:
# Training & evaluation functions

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct = 0.0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device).unsqueeze(1)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += ((torch.sigmoid(logits) >= 0.5).float() == labels).sum().item()
    n = len(loader.dataset)
    return total_loss / n, correct / n


def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct = 0.0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device).unsqueeze(1)
            logits = model(imgs)
            loss   = criterion(logits, labels)
            total_loss += loss.item() * imgs.size(0)
            correct    += ((torch.sigmoid(logits) >= 0.5).float() == labels).sum().item()
    n = len(loader.dataset)
    return total_loss / n, correct / n

In [ ]:
# Training Loop
import shutil

DRIVE_PATH = '/content/drive/MyDrive/realface/checkpoints/'
os.makedirs(DRIVE_PATH, exist_ok=True)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

history    = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0
TOTAL_EPOCHS = 10

for epoch in range(1, TOTAL_EPOCHS + 1):

    # --- fase 1: freeze backbone, fase 2: unfreeze semua ---
    if epoch == 1:
        for name, param in model.named_parameters():
            param.requires_grad = 'classifier' in name
        for g in optimizer.param_groups:
            g['lr'] = 1e-3
        print('Phase 1: classifier only  (epoch 1-3)')

    elif epoch == 4:
        for param in model.parameters():
            param.requires_grad = True
        for g in optimizer.param_groups:
            g['lr'] = 1e-4
        print('Phase 2: full fine-tune  (epoch 4-10)')

    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss,   val_acc   = eval_epoch(model,  val_loader,   criterion, device)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    print(f'Epoch {epoch:02d}/{TOTAL_EPOCHS}  '
          f'train loss: {train_loss:.4f}  acc: {train_acc:.4f}  |  '
          f'val loss: {val_loss:.4f}  acc: {val_acc:.4f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        ckpt_path = os.path.join(DRIVE_PATH, 'best_model.pth')
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'val_acc': val_acc}, ckpt_path)
        print(f'  -> checkpoint saved  (val_acc: {val_acc:.4f})')

print(f'\nBest val acc: {best_val_acc:.4f}')

In [ ]:
# Plot Training History
epochs = range(1, len(history['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs, history['train_loss'], label='Train Loss')
ax1.plot(epochs, history['val_loss'],   label='Val Loss')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend()

ax2.plot(epochs, history['train_acc'], label='Train Acc')
ax2.plot(epochs, history['val_acc'],   label='Val Acc')
ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch'); ax2.legend()

plt.tight_layout()
save_path = os.path.join(DRIVE_PATH, 'training_history.png')
plt.savefig(save_path, dpi=150)
plt.show()
print(f'Plot saved to {save_path}')

In [ ]:
# Load Best Model & Evaluasi Test Set
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

ckpt = torch.load(os.path.join(DRIVE_PATH, 'best_model.pth'), map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

y_true, y_pred, y_prob = [], [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs   = imgs.to(device)
        logits = model(imgs).squeeze(1)
        probs  = torch.sigmoid(logits).cpu().numpy()
        preds  = (probs >= 0.5).astype(int)
        y_prob.extend(probs.tolist())
        y_pred.extend(preds.tolist())
        y_true.extend(labels.int().tolist())

print(f'Accuracy  : {accuracy_score(y_true, y_pred):.4f}')
print(f'Precision : {precision_score(y_true, y_pred):.4f}')
print(f'Recall    : {recall_score(y_true, y_pred):.4f}')
print(f'F1-Score  : {f1_score(y_true, y_pred):.4f}')

In [ ]:
# Confusion Matrix
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Fake', 'Real'],
            yticklabels=['Fake', 'Real'], ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix')

plt.tight_layout()
cm_path = os.path.join(DRIVE_PATH, 'confusion_matrix.png')
plt.savefig(cm_path, dpi=150)
plt.show()
print(f'Saved to {cm_path}')

In [ ]:
# ROC Curve
from sklearn.metrics import roc_curve, roc_auc_score

fpr, tpr, _ = roc_curve(y_true, y_prob)
auc_score   = roc_auc_score(y_true, y_prob)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, label=f'AUC = {auc_score:.4f}')
ax.plot([0, 1], [0, 1], 'k--')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve'); ax.legend()

plt.tight_layout()
roc_path = os.path.join(DRIVE_PATH, 'roc_curve.png')
plt.savefig(roc_path, dpi=150)
plt.show()
print(f'Saved to {roc_path}')

In [ ]:
# Error Analysis
import numpy as np

y_true_arr = np.array(y_true)
y_pred_arr = np.array(y_pred)
y_prob_arr = np.array(y_prob)

fp_idx = np.where((y_pred_arr == 1) & (y_true_arr == 0))[0][:5]  # prediksi Real, asli Fake
fn_idx = np.where((y_pred_arr == 0) & (y_true_arr == 1))[0][:5]  # prediksi Fake, asli Real

def plot_errors(indices, title):
    n = len(indices)
    if n == 0:
        print(f'{title}: tidak ada contoh'); return
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3))
    if n == 1: axes = [axes]
    fig.suptitle(title, fontsize=11)
    for ax, idx in zip(axes, indices):
        ax.imshow(Image.open(X_test[idx]).convert('RGB'))
        true_lbl = 'Real' if y_true_arr[idx] == 1 else 'Fake'
        pred_lbl = 'Real' if y_pred_arr[idx] == 1 else 'Fake'
        conf     = y_prob_arr[idx] if y_pred_arr[idx] == 1 else 1 - y_prob_arr[idx]
        ax.set_title(f'True: {true_lbl}\nPred: {pred_lbl} ({conf:.2f})', fontsize=8)
        ax.axis('off')
    plt.tight_layout(); plt.show()

plot_errors(fp_idx, 'False Positives  (prediksi Real, asli Fake)')
plot_errors(fn_idx, 'False Negatives  (prediksi Fake, asli Real)')

In [ ]:
# Export ONNX
import onnx
import onnxruntime as ort

model.eval()
dummy_input = torch.randn(1, 3, 224, 224, device=device)
onnx_path   = os.path.join(DRIVE_PATH, 'realface.onnx')

torch.onnx.export(
    model, dummy_input, onnx_path,
    opset_version=11,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)
onnx.checker.check_model(onnx_path)
print(f'ONNX model saved to {onnx_path}')

sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
out  = sess.run(None, {'input': dummy_input.cpu().numpy()})
print(f'ONNX inference output shape: {out[0].shape}')

In [ ]:
# Download semua output ke lokal
from google.colab import files

outputs = [
    os.path.join(DRIVE_PATH, 'realface.onnx'),
    os.path.join(DRIVE_PATH, 'best_model.pth'),
    os.path.join(DRIVE_PATH, 'confusion_matrix.png'),
    os.path.join(DRIVE_PATH, 'roc_curve.png'),
    os.path.join(DRIVE_PATH, 'training_history.png'),
]

for path in outputs:
    if os.path.exists(path):
        files.download(path)
        print(f'Downloaded: {os.path.basename(path)}')
    else:
        print(f'Not found:  {os.path.basename(path)}')